# Module `algorithms/tabu_search.py`

La recherche tabou explore **tout** le voisinage (best-improvement global), accepte le meilleur mouvement meme s'il degrade, et interdit pendant quelques iterations (le *tenure*) les mouvements recemment faits pour empecher les cycles.

- Attribut tabou relocate : **client** deplace.
- Attribut tabou swap : **paire** de clients echangee.
- **Aspiration** : un mouvement tabou est autorise s'il ameliore strictement le meilleur cout connu.


In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent.parent / "src"))

from cesipath import GraphGenerationConfig, GraphGenerator
from cesipath.algorithms import tabu_search
from cesipath.solver_input import build_static_solver_input


## 1. Appel standard


In [2]:
cfg = GraphGenerationConfig(node_count=18, seed=5)
instance = GraphGenerator(cfg).generate()
si = build_static_solver_input(instance)

sol = tabu_search(si, max_iterations=200, tabu_tenure=7, seed=0)
print(f"cout = {sol.total_cost:.2f}  ; routes = {sol.route_count}")


cout = 764.32  ; routes = 4


## 2. Parametres

| Parametre | Role |
|---|---|
| `max_iterations` | Borne dure sur le nombre d'iterations. |
| `tabu_tenure` | Duree pendant laquelle un attribut reste tabou. Plus grand = plus diversifiant, moins intensif. |
| `max_no_improve` | Arret precoce si aucun progres depuis N iterations. |
| `initial_rcl_alpha` | Construction initiale : 0 = plus proche voisin, >0 = randomise. |
| `final_local_search` | Descente deterministe en post-traitement (recommande). |


## 3. Effet du `tabu_tenure`

Un tenure tres faible laisse l'algo tourner en rond autour d'un optimum local ; un tenure trop grand l'empeche de polir une bonne piste. En pratique une valeur entre `sqrt(n)` et `n/2` fonctionne bien.


In [3]:
for tenure in (0, 3, 7, 15, 40):
    sol = tabu_search(si, max_iterations=150, tabu_tenure=tenure, max_no_improve=150, seed=1)
    print(f"tenure={tenure:>2} -> cout={sol.total_cost:.2f}  routes={sol.route_count}")


tenure= 0 -> cout=797.41  routes=4
tenure= 3 -> cout=784.29  routes=4
tenure= 7 -> cout=764.32  routes=4
tenure=15 -> cout=784.29  routes=4
tenure=40 -> cout=764.32  routes=4


## 4. Arret precoce via `max_no_improve`

Si l'algo n'a pas trouve mieux depuis `max_no_improve` iterations, on s'arrete. C'est utile pour ne pas gaspiller du budget sur un plateau.


In [4]:
import time

for cap in (20, 80, 500):
    t0 = time.perf_counter()
    sol = tabu_search(si, max_iterations=500, tabu_tenure=7, max_no_improve=cap, seed=2)
    dt = time.perf_counter() - t0
    print(f"max_no_improve={cap:>3} -> cout={sol.total_cost:.2f}  temps={dt*1000:.1f} ms")


max_no_improve= 20 -> cout=764.32  temps=9.3 ms
max_no_improve= 80 -> cout=764.32  temps=17.8 ms
max_no_improve=500 -> cout=764.32  temps=63.3 ms


## 5. Aspiration : debloquer un mouvement tabou

Lorsqu'un mouvement tabou produirait un cout **strictement** meilleur que le meilleur connu, il est autorise. Sans aspiration, l'algo pourrait s'interdire de visiter un nouvel optimum global par accident. C'est implemente dans les deux blocs de selection (relocate et swap) via la condition `neighbor_cost < best_cost - 1e-9`.


## 6. Comparaison rapide avec GRASP et SA

Ces trois algos partagent la meme recherche locale finale. Le tabou explore tout le voisinage a chaque iteration -> plus couteux par iteration mais souvent plus precis sur des instances petites/moyennes.


In [5]:
from cesipath.algorithms import grasp, simulated_annealing

import time

def run(name, fn, **kwargs):
    t0 = time.perf_counter()
    sol = fn(si, seed=0, **kwargs)
    dt = time.perf_counter() - t0
    print(f"{name:<6} cout={sol.total_cost:>7.2f}  temps={dt*1000:>7.1f} ms")

run("grasp",  grasp,              max_iterations=30, rcl_alpha=0.3)
run("sa",     simulated_annealing, max_iterations=2000, cooling_rate=0.97)
run("tabu",   tabu_search,         max_iterations=150, tabu_tenure=7)


grasp  cout= 764.32  temps=   19.6 ms
sa     cout= 797.41  temps=    4.3 ms
tabu   cout= 764.32  temps=   11.5 ms


## 7. Reproductibilite


In [6]:
a = tabu_search(si, max_iterations=80, tabu_tenure=7, seed=9)
b = tabu_search(si, max_iterations=80, tabu_tenure=7, seed=9)
print("identiques ?", a.total_cost == b.total_cost and a.routes == b.routes)


identiques ? True
